# Officer of the Watch (OOW) Agent — From COLREG Text to Agentic Training Data
## Auto Pilot Project · OOW Module

This notebook is the **OOW analogue of `VHF_Agent_Training_Pipeline.ipynb`**: same pipeline shape
(§0 setup → §8 JSON → §9 RAG → §10 KG → §11 reasoning traces → §12 SFT/DPO/reflection → §13 quality
gates → §14 fine-tune → §15 merge → §16 eval → §17-20 compression), pointed at a different domain via
`AUTOPILOT_DOMAIN=OOW` instead of hardcoding a second copy of every script.

### What the OOW agent is (see `Docs/Multi-Agent_Maritime_Navigation_System_Architecture_Plan rev JS v4.pdf`)

In the multi-agent ship's-bridge architecture, the **Officer of the Watch (= "Navigation Agent")**:
- receives parsed contact tracks from the **Perception/Lookout agent** (bearing, range, CPA/TCPA,
  encounter type) and own-ship position/state from the **Localization/Quartermaster agent** — both in
  natural language, not raw sensor data;
- runs pathfinding (A*/DWA) and **strictly applies COLREG** to decide the collision-avoidance action;
- outputs a **human-style helm + engine order** ("30 degrees port", "50 degrees starboard", "1 degree
  port"; engine "full ahead" / "full back" / "continue course") — **not** a raw MOOS `DESIRED_HEADING`/
  `DESIRED_SPEED` call. Translating that order into an exact rudder angle/RPM is the job of a separate,
  later **Actuator Agent (Helmsman)**, per the architecture doc's own phasing.
- Per the architecture doc §8 ("Language Models" / data-per-agent table): the Navigation Agent is
  fine-tuned on **COLREG + maritime training handbooks**; the Actuator/Perception/Localization agents
  need **no fine-tuning** (interface/translation layers only) — so this notebook is the one place real
  training work happens for this whole sub-tree.

### Learning goals
- Reuse the *exact same* generic pipeline scripts as VHF (`pipeline/ingest`, `pipeline/track1`,
  `pipeline/train`, `pipeline/eval`, `pipeline/compress`), selected via the `AUTOPILOT_DOMAIN` env var
  (`core.AgentPaths.from_env()`), rather than forking a parallel copy of every script.
- Understand why COLREG's Track 1 (rules & knowledge) is fully built here, while Track 2 (applied
  helm/engine-order decision-making) is deliberately scoped as **Phase 2** (see §12.5/§12.6 below) —
  analogous to how VHF's Track 2 (conversational compliance) was built as a second pass after Track 1
  was working end-to-end.

## Two tracks, from day one

| | Track 1 — COLREG rules & knowledge | Track 2 — Applied helm/engine-order decisions |
|---|---|---|
| Eval data (held out, NEVER used for training) | `Data/OOW/OOW_Eval/colreg_qa_500.json` (500 Q&A) | *(planned)* scenario → helm/engine-order pairs derived from the existing 12 MOOS scenario geometries (`generate_moos_scenarios.py` / `scenarios_manifest.json`) |
| Training data | `oow_sft_direct/cot/rag.jsonl`, `oow_multihop.jsonl`, `oow_dpo_pairs.jsonl`, `oow_reflection.jsonl` | *(Phase 2 — not yet built, see §12.5/§12.6)* |
| Eval script | `pipeline/eval/eval_finetuned.py --gold-file colreg_qa_500.json` | *(Phase 2)* |

Both tracks would feed the **same** QLoRA fine-tune, exactly like VHF's `train_sft.py`/`train_dpo.py`/
`train_reflection.py` load a list of files spanning both tracks — but this notebook only builds
Track 1 end-to-end for now. Track 2 needs its own scenario/category design (analogous to
`pipeline/track2/build_vhf_colreg_scenarios.py`) before it can be generated; see §12.5/§12.6 for the
concrete plan.

## Repository layout — where OOW data & models live

```
Data/OOW/                          ← AgentPaths.oow().data_root
├── OOW_Protocols/                 ← source_dir   (training documents: COLREG-Consolidated-2018.pdf, ...)
├── OOW_Eval/                      ← eval_dir     (held-out evaluation)
│   └── colreg_qa_500.json         ← gold Q&A (500 items) — NOT the default oow_gold_answers.json name,
│                                     loaded explicitly via --gold-file everywhere in this notebook
├── OOW_JSON/                      ← json_dir     (§ 8 output)
├── OOW_Agents_Training/           ← cache_dir    (RAG/KG/traces/SFT/DPO/reflection)
├── OOW_Literature_Review/         ← background research (not a pipeline input)
└── OOW_MOOS_Integration/          ← colreg_llm_bridge.py + MOOS scenario/eval scaffolding (Track 2 input, later)
_models/OOW/                       ← domain_models_dir (adapters + merged OOW-QWEN)
```

This mirrors VHF's layout exactly (see that notebook's own "Repository layout" section) — only the
`Data/<domain>` segment and the two domain-specific filenames (`OOW_Protocols`, `colreg_qa_500.json`)
differ. Every path below comes from `AgentPaths.oow()` (equivalently, `AgentPaths.from_env()` with
`AUTOPILOT_DOMAIN=OOW`) — never hardcoded — so this notebook and the shared pipeline scripts can never
silently diverge on where things live.

---
## § 0 — Environment Setup

Same dependency set as the VHF notebook (this is a workspace-wide `.venv`, not domain-scoped) — safe to
skip if you already ran VHF's § 0 in this environment.

In [ ]:
import subprocess, sys, time

packages = [
    # Core inference (API clients)
    "openai>=1.40.0",
    "anthropic>=0.34.0",
    "tiktoken>=0.7.0",
    # Retrieval / embeddings
    "sentence-transformers>=2.7.0",
    "scikit-learn>=1.4.0",
    # Data
    "pandas>=2.2.0",
    "numpy>=1.26.0",
    "networkx>=3.2",
    # PDF extraction
    "pdfplumber>=0.11.0",
    # Progress + plotting
    "tqdm>=4.66.0",
    "matplotlib>=3.8.0",
    # HuggingFace stack (torch is installed separately in § 0.2 with CUDA support)
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "trl>=0.8.0",
    "accelerate>=0.29.0",
]

t0 = time.time()
print("[setup] Installing / verifying dependencies (excluding torch)...", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print(f"[setup] Done in {time.time()-t0:.1f}s", flush=True)

print("\n[setup] Installing bitsandbytes (for 4-bit Qwen quantisation)...", flush=True)
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.43.0"])
    print("[setup] bitsandbytes installed OK")
except subprocess.CalledProcessError as e:
    print(f"[setup] bitsandbytes install failed ({e}). Qwen will load in bf16 instead of 4-bit.")

### § 0.1 — Domain selection

Every subsequent cell in this notebook — and every `pipeline.*` subprocess it launches — targets the
**OOW** domain via this one environment variable (`core.AgentPaths.from_env()` reads it). This is the
only line that differs from an equivalent VHF cell.

In [ ]:
import os
from pathlib import Path
from core import AgentPaths

os.environ["AUTOPILOT_DOMAIN"] = "OOW"

paths = AgentPaths.oow()
W = paths.workspace
GOLD_FILE = paths.eval_file("colreg_qa_500.json")   # NOT paths.gold_file -- see repo-layout note above

print(paths.describe())
print(f"\ngold Q&A file exists: {GOLD_FILE.exists()}  ({GOLD_FILE})")

### § 0.2 — CUDA-aware PyTorch + GPU sanity check

Identical to VHF's § 0.1/§ 0.2 — copy verbatim since this is workspace-level, not domain-scoped.

In [ ]:
import subprocess, sys, shutil, time

def _detect_nvidia_gpu() -> tuple[bool, str]:
    if shutil.which("nvidia-smi") is None:
        return False, "nvidia-smi not found on PATH"
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
             "--format=csv,noheader"],
            text=True, timeout=10,
        ).strip()
        return True, out
    except Exception as e:
        return False, f"nvidia-smi failed: {e}"

has_gpu, gpu_info = _detect_nvidia_gpu()
print(f"[gpu-detect] {gpu_info}")

try:
    import torch
    need_reinstall = has_gpu != torch.cuda.is_available()
except ImportError:
    need_reinstall = True

if need_reinstall:
    index_url = "https://download.pytorch.org/whl/cu124" if has_gpu else "https://download.pytorch.org/whl/cpu"
    t0 = time.time()
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "torch", "torchvision", "torchaudio", "--index-url", index_url,
    ])
    print(f"[install] Done in {time.time()-t0:.1f}s -- RESTART THE KERNEL, then re-run.")
else:
    print("[install] Skipped: existing torch already matches the desired CUDA state.")

In [ ]:
import torch
print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}  ({props.total_memory/1024**3:.1f} GB)")
    try:
        import bitsandbytes as bnb
        print(f"bitsandbytes {bnb.__version__} OK")
    except Exception as e:
        print(f"bitsandbytes not available ({e}) -- Qwen will load in bf16 instead of 4-bit.")
else:
    print("No CUDA -- fine-tuning/eval will fall back to CPU (very slow). See VHF notebook § 0.2 for fixes.")

---
## § 1 — Load the existing COLREG gold Q&A set

Unlike VHF (§1-3: parse raw exam text, then LLM-generate gold answers), OOW's Track 1 eval set
(`colreg_qa_500.json`) **already exists**, hand-built with schema
`{id, category, rule_ref, question_type, question, answer, explanation, difficulty}` spanning COLREG
Parts A-E and Annexes I-IV (see `Data/OOW/OOW_Eval/README.md`). We just load and sanity-check it here;
no generation step is needed. Note the field is `answer`, not `gold_answer` as in VHF's schema --
downstream scripts that expect `gold_answer`/`expected_points` (contamination filters, `eval_finetuned.py`)
need this file normalised first, which the cell below does into a `*_normalised.json` copy.

In [ ]:
import json

raw_gold = json.loads(GOLD_FILE.read_text(encoding="utf-8"))
print(f"Loaded {len(raw_gold)} COLREG Q&A records from {GOLD_FILE.name}")
print("Sample record keys:", list(raw_gold[0].keys()))

# Normalise to the {question, gold_answer, expected_points} shape every shared
# pipeline script (contamination filters, eval_finetuned.py) expects.
NORM_FILE = paths.eval_dir / "colreg_qa_500_normalised.json"
normalised = [
    {
        "id":               r["id"],
        "section_id":       r.get("rule_ref", r.get("category", "")),
        "section_title":    r.get("category", ""),
        "type":             r.get("question_type", ""),
        "question":         r["question"],
        "gold_answer":      r.get("answer", ""),
        "expected_points":  [r["explanation"]] if r.get("explanation") else [],
    }
    for r in raw_gold
]
NORM_FILE.write_text(json.dumps(normalised, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote normalised copy: {NORM_FILE}  ({len(normalised)} rows)")

import pandas as pd
df = pd.DataFrame(raw_gold)
if "category" in df.columns:
    print(df["category"].value_counts())

---
## § 5-7 — Baseline evaluation: Qwen2.5-7B (base, no fine-tuning)

Instead of duplicating VHF's in-kernel Tutorial-13 metrics engine, we call the already-generic
`pipeline/eval/eval_finetuned.py` directly (it has its own SemSim/AnsRel/Faith/Correct/Cover/NumHit/
LitHit metrics and a `--gold-file` override we added specifically for this). Needs `OPENAI_API_KEY` in
`.env` for the Faith/Correct judge metrics.

In [ ]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-X", "utf8", "-m", "pipeline.eval.eval_finetuned",
    "--model", "Qwen/Qwen2.5-7B-Instruct",
    "--tag", "oow_qwen_base",
    "--gold-file", str(NORM_FILE),
    # "--n", "50",  # uncomment for a smoke test first
], cwd=W)

In [ ]:
import json
import matplotlib.pyplot as plt

summary = json.loads((paths.cache_dir / "eval_oow_qwen_base_summary.json").read_text())
means = {k: v for k, v in summary["means"].items() if v is not None}
plt.figure(figsize=(8, 4))
plt.bar(means.keys(), means.values())
plt.title(f"OOW baseline (qwen_base) -- Track 1, n={summary['n']}")
plt.ylabel("score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
print(json.dumps(summary["means"], indent=2))

---
## § 8 — Convert COLREG source documents to structured JSON

VHF's § 8 lives entirely as in-notebook cells because it has to cope with ~30 heterogeneous documents.
OOW's Track 1 source is a single, very regularly structured official text, so this is a small
standalone script instead: `pipeline/ingest/build_oow_json.py` (Convention Articles I-IX → PART A-E /
Rule 1-38 → Annexes I-IV, ~105 sections). See that script's docstring for the exact schema.

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.ingest.build_oow_json"], cwd=W)

---
## § 9 — Build RAG chunks from JSON

Same chunker as VHF (`pipeline/ingest/build_rag.py`), unchanged -- just reads from `OOW_JSON/` and
writes `oow_rag_chunks.json` / `oow_rag_embeddings.npy` / `oow_rag_chunk_ids.json` into
`OOW_Agents_Training/` instead of the `vhf_*` equivalents.

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.ingest.build_rag"], cwd=W)

---
## § 10 — Knowledge-Graph RAG

Same KG builder as VHF (`pipeline/ingest/build_kg.py`). Note: `CONCEPT_ALIASES` inside that module is
still VHF-flavoured (channel numbers, prowords) -- it degrades gracefully for COLREG (just contributes
no alias hits) but a COLREG-specific alias table (rule numbers, "give-way"/"stand-on", light/shape
terms) would improve retrieval quality; tracked as a follow-up, not blocking Track 1.

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.ingest.build_kg"], cwd=W)

---
## § 11 — Reasoning-trace extraction

Same extraction driver as VHF (`pipeline/track1/extract_reasoning.py`), with a COLREG-specific
`SYSTEM_PROMPT` variant selected automatically by `paths.domain`. It reuses VHF's exact trace schema
field names for mechanical compatibility with every downstream script -- see the prompt for the
mapping (`channels` → COLREG rule numbers, `prowords_used` → vessel-role/situational tags,
`regulations` → instruments like "COLREG 1972"). Needs `OPENAI_API_KEY`; resume-safe (only calls the
LLM for chunk_ids missing from the output file).

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.track1.extract_reasoning"], cwd=W)

---
## § 11.5 — Real-world incident reports (screen → excerpt → extract)

`Data/OOW/OOW_Incidents/` has ~800 real marine accident-investigation reports (UK MAIB + US
NTSB/USCG). Most are groundings/fires/flooding/machinery-failure cases with no COLREG angle, so we
can't just feed the whole corpus in. Three-stage local pipeline, mirroring § 8-11 above but for
incidents instead of the rule text:

1. **`pipeline/ingest/screen_incidents.py`** — scores every PDF on COLREG-vocabulary density minus
   non-COLREG signal (grounding/fire/flooding/...), writes a ranked
   `incident_screening.json` (no LLM, just keyword counting).
2. **`pipeline/ingest/build_incident_excerpts.py`** — for reports scoring `net_score >= 15`, keeps only
   the report's opening summary + its Analysis/Conclusions/Findings pages (not the whole report),
   saved as `Data/OOW/OOW_JSON/incident_*.json` in the same schema as `colreg_consolidated_2018.json`
   so § 9/§ 10 (RAG/KG) would pick them up unchanged if re-run.
3. **`pipeline/track1/extract_incident_reasoning.py`** — GPT-4o-mini extraction, ONE trace per
   incident document (not per chunk, since each excerpt is already a single bounded unit). Schema
   reuses the § 11 field names (`situation`/`procedures`/`channels`/... ) for mechanical
   compatibility with every § 12 builder below, plus an additive `incident` sub-object (vessels/
   roles, fault attribution, actual-vs-correct actions, exceptional circumstances, confidence). The
   model also self-filters: reports that turn out NOT to be vessel-vs-vessel collision-avoidance
   situations return `{"skip": true, ...}` instead of a trace.


In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.ingest.screen_incidents"], cwd=W)

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.ingest.build_incident_excerpts",
                       "--min-score", "15"], cwd=W)

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.track1.extract_incident_reasoning"], cwd=W)

---
## § 12 — Track 1 training-data generation (deterministic, no LLM required)

Same four builders as VHF's Track 1, called with `--out-prefix oow_sft` / `--gold-file` pointed at the
normalised COLREG Q&A so contamination filtering works correctly.

In [ ]:
subprocess.check_call([
    sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_sft",
    "--out-prefix", "oow_sft", "--gold-file", str(NORM_FILE),
], cwd=W)

In [ ]:
subprocess.check_call([
    sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_multihop",
    # Both traces files together: rule-text traces all share ONE source_file
    # (COLREG-Consolidated-2018.pdf) so on their own they can never satisfy
    # build_multihop.py's "different source documents" requirement -- adding
    # the 67 incident traces (each its own source_file) is what unlocks real
    # cross-track pairs (rule text <-> real incident excerpt).
    "--traces-file", str(paths.cache_dir / "oow_reasoning_traces.jsonl"),
                     str(paths.cache_dir / "oow_incident_reasoning_traces.jsonl"),
    "--out-file", str(paths.cache_dir / "oow_multihop.jsonl"), "--gold-file", str(NORM_FILE),
], cwd=W)

In [ ]:
subprocess.check_call([
    sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_rlhf",
    "--out-file", str(paths.cache_dir / "oow_dpo_pairs.jsonl"), "--gold-file", str(NORM_FILE),
], cwd=W)

In [ ]:
subprocess.check_call([
    sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_reflection",
    "--out-file", str(paths.cache_dir / "oow_reflection.jsonl"), "--gold-file", str(NORM_FILE),
], cwd=W)

---
## § 12.7 — Track 1 training data from real incident excerpts

Same four deterministic builders, pointed at `oow_incident_reasoning_traces.jsonl` (§ 11.5) instead
of the rule-text traces, with their own `oow_incident_*` output files (kept separate from `oow_sft_*`
etc. for traceability, same convention as VHF's Track 1 / Track 2 split). Cross-track multi-hop pairs
(rule text <-> incident) are already produced by the § 12 `build_multihop` cell above, since it's
pointed at both traces files together -- nothing to do here for multi-hop.


In [ ]:
subprocess.check_call([
    sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_sft",
    "--traces-file", str(paths.cache_dir / "oow_incident_reasoning_traces.jsonl"),
    "--out-prefix", "oow_incident_sft", "--gold-file", str(NORM_FILE),
], cwd=W)

In [ ]:
subprocess.check_call([
    sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_rlhf",
    "--traces-file", str(paths.cache_dir / "oow_incident_reasoning_traces.jsonl"),
    "--out-file", str(paths.cache_dir / "oow_incident_dpo_pairs.jsonl"), "--gold-file", str(NORM_FILE),
], cwd=W)

In [ ]:
subprocess.check_call([
    sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_reflection",
    "--traces-file", str(paths.cache_dir / "oow_incident_reasoning_traces.jsonl"),
    "--out-file", str(paths.cache_dir / "oow_incident_reflection.jsonl"), "--gold-file", str(NORM_FILE),
], cwd=W)

In [ ]:
# Row-count sanity check across everything § 12 / § 12.7 produced.
for name in ["oow_sft_direct.jsonl", "oow_sft_cot.jsonl", "oow_sft_rag.jsonl",
             "oow_multihop.jsonl", "oow_dpo_pairs.jsonl", "oow_reflection.jsonl",
             "oow_incident_sft_direct.jsonl", "oow_incident_sft_cot.jsonl", "oow_incident_sft_rag.jsonl",
             "oow_incident_dpo_pairs.jsonl", "oow_incident_reflection.jsonl"]:
    p = paths.cache_dir / name
    n = sum(1 for _ in p.open(encoding="utf-8")) if p.exists() else 0
    print(f"  {name:<30} {n:>5} rows" if p.exists() else f"  {name:<30} MISSING")

---
## § 12.9 — Gold-standard file + atomic-claim enrichment (metric suite v2 prerequisite)

Every evaluation below (§13 ablation scoring, §16, §20) defaults to the **claim-level RAGAS suite v2**
([ragas_metrics.py](pipeline/eval/ragas_metrics.py)) — same suite, same composites and same
paired-bootstrap CIs as VHF (see the VHF notebook §12.9/§13/§16 for the full metric table and the
rationale for replacing the old whole-answer binary Faith judge). The suite needs each gold record
decomposed into atomic `gold_claims` — this section documents the required gold format and runs the
one-off enrichment for OOW.

### How the OOW gold-standard file must look

`Data/OOW/OOW_Eval/colreg_qa_500_normalised.json` (500 records, **held out** — nothing derived from it
may ever enter training data; every builder cosine-filters against it at 0.85):

```json
{"id": "colreg_0001",
 "section_id": "Rule 15",            ← rule reference (used for gap analysis grouping)
 "section_title": "Crossing situation",
 "type": "recall | scenario | why",
 "question": "…one exam-style question…",
 "gold_answer": "…terse but fluent reference answer…",
 "expected_points": ["…atomic examinable point(s)…"]}
```

### The `gold_claims` enrichment (sibling `*_claims.json` file)

[enrich_gold_claims.py](pipeline/eval/enrich_gold_claims.py) has **Claude** (not GPT — the gold was
authored with Claude, and the GPT-4o-mini judge must never grade its own model family's annotations)
decompose each record into 3-10+ atomic claims, appended as one extra `gold_claims` field:

```json
"gold_claims": [
  {"claim": "Rule 15 governs crossing situations between power-driven vessels",
   "type": "number", "value": "15", "role": "colreg_rule"},
  {"claim": "The vessel which has the other on her starboard side keeps out of the way",
   "type": "procedure"},
  {"claim": "The give-way vessel avoids crossing ahead of the other vessel",
   "type": "procedure"},
  {"claim": "Own vessel alters course to starboard",
   "type": "direction", "value": "starboard", "role": "turn_direction"}
]
```

Rules (enforced by [gold_claims.py](pipeline/eval/gold_claims.py)'s validator, `--check` CLI): atomic &
self-contained; grounded ONLY in the record; every `expected_points` entry covered by ≥1 claim; every
safety-critical number a `type:"number"` claim with a role from the fixed vocabulary (for OOW mostly
`colreg_rule`, `distance_nm`, `time_interval`, plus `turn_direction`/`pass_side`/`light_colour`
direction claims); internal gold inconsistencies get a `QA_FLAG` claim for human review — on the VHF
scenario file this mechanism surfaced 59 flags including ~15 candidate genuine COLREG errors, so read
the flags, don't just count them.


In [ ]:
# § 12.9 code: enrich the OOW gold file with atomic claims (Anthropic API,
# needs ANTHROPIC_API_KEY in .env), then validate. Resume-safe: rerunning only
# processes records that are missing or were rejected by the validator.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.enrich_gold_claims",
                       "--files", str(NORM_FILE)], cwd=W)
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.gold_claims", "--check",
                       str(NORM_FILE.with_name(NORM_FILE.stem + "_claims.json"))], cwd=W)


---
## § 12.10 — Procedural Graph: *what-to-do* naast de KG's *what-is* (Lu et al. 2026)

Zie VHF-notebook § 12.10 voor de volledige uitleg. Kort: [build_pg.py](pipeline/ingest/build_pg.py)
mint (procedure, NEXT, procedure)-triplets uit de geordende `procedures` in de reasoning-traces
(edge-attributen: constraints→condition, step-why→guidance, warnings→pitfalls), deterministisch met
een merge-guard op kanaalnummers/port-starboard. Afnemers: de `v4_pg` ablation-config (§ 13, guidance
in de prompt), `oow_pg_sft.jsonl` step-order trainingsdata (gewired in `train_sft.py`), de zesde
DPO-as `swap_step_order`, en de `ProcOrder`-metric in de suite (gerapporteerd, nog niet in de composite).

**OOW-kanttekening**: COLREG-regeltekst is grotendeels declaratief — slechts 31 traces leveren nu
geordende stappen (~125 nodes / 100 edges), dus de OOW-PG is voorlopig een skelet. Hij groeit
automatisch mee zodra Track 2 (helm/engine-orders, § 12.5/12.6) en meer incident-traces bestaan;
[evolve_pg.py](pipeline/train/evolve_pg.py) (fase C, self-evolution met validation-gating) wordt pas
zinvol in de MOOS-in-the-loop fase.


In [ ]:
# § 12.10 code: Procedural Graph + step-order trainingsdata voor OOW.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.ingest.build_pg"], cwd=W)
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_pg_sft",
                       "--gold-file", str(NORM_FILE)], cwd=W)


---
## § 13 — Prompt ablation: RAG/CoT-in-the-prompt vs. plain base (before any fine-tuning)

Before spending GPU time on fine-tuning, check how much of the available Track 1 knowledge can be
captured just by **prompting** the base model with retrieved context and/or a chain-of-thought
instruction -- no weights touched. This is the same V0/V1/V2/V3 design as VHF's own §13 ablation
(`pipeline/eval/prep_ablation.py` / `run_ablation.py` / `score_ablation.py`, now domain-parametrized
via `AgentPaths.from_env()` exactly like the rest of the pipeline):

- **v0_base** — plain question, no context, no reasoning instruction (same as the §5-7 baseline).
- **v1_rag** — the top-k retrieved RAG chunks (built in §9) injected as context.
- **v2_cot** — a chain-of-thought instruction, no retrieved context.
- **v3_rag_cot** — both together.
- **v4_pg** — a "Procedure guidance" block rendered from the Procedural Graph (§ 12.10) in the
  prompt; questions without a renderable path fall back to the v0 prompt at run time.

**Scoring uses the claim-level RAGAS suite v2** (see VHF notebook §13 for the full metric table):
the RAG configs get real RAGAS Faithfulness + ContextPrecision/ContextRecall (contexts recovered from
`ablation_prompts.json`), the closed-book configs get CorpusGrounded; AnswerCorrectness (claim F1 vs
§12.9's `gold_claims`), AnswerRelevancy, role-aware NumericF1, LitHit, Cover and ProcOrder (reported,
not in the composite) apply everywhere.
Composite weights are fixed per config kind and the summary includes paired-bootstrap 95% CIs of each
config vs v0_base. Requires §12.9's `colreg_qa_500_normalised_claims.json` — pass it via
`--gold-file`; `--legacy` reproduces the old v1 metrics for comparison with historical numbers.

For VHF, this ordering turned out to matter a lot: v3_rag_cot (0.511) beat both the plain base (0.476)
**and** every fine-tuned checkpoint tried so far (0.24-0.31, before the `assistant_only_loss` fix) --
strong evidence that the underlying knowledge is genuinely useful, and that problems seen after
fine-tuning are about *how* that knowledge gets baked into weights, not whether the knowledge itself
is any good. Running this for OOW first, before committing to §14, gives the same sanity check here.
(Historical note: the n=40 COLREG-only ablation preserved in `ablation_*_colreg_only_n40.*` was scored
with the old `legacy_v1` suite — don't compare those numbers against `ragas_v2.0` summaries.)


In [ ]:
subprocess.check_call([
    sys.executable, "-X", "utf8", "-m", "pipeline.eval.prep_ablation",
    "--n", "40", "--gold-file", str(NORM_FILE),
], cwd=W)

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.run_ablation"], cwd=W)

In [ ]:
# New-suite scoring needs the gold_claims sibling of the OOW gold file (§12.9).
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.score_ablation",
                       "--gold-file", str(NORM_FILE)], cwd=W)

import json
summary = json.loads((paths.cache_dir / "ablation_summary.json").read_text())
print(f"metric_suite: {summary.get('_meta', {}).get('metric_suite', 'legacy_v1')}")
for cfg, means in summary.items():
    if cfg == "_meta":
        continue
    ci = means.get("Composite_vs_v0_base")
    ci_str = (f"  Δvs_v0={ci['delta']:+.3f} [{ci['ci_lo']:+.3f},{ci['ci_hi']:+.3f}]"
              f"{' *' if ci.get('significant') else ' (noise)'}" if ci else "")
    print(f"{cfg:<12} Composite={means['Composite']:.3f}{ci_str}")


---
## § 12.5 / § 12.6 — Track 2: applied helm/engine-order decisions (Phase 2 — not yet built)

This is the piece the user's spec calls out explicitly: given a fused situation report (contacts with
bearing/range/CPA/TCPA/encounter-type from the Lookout agent + own-ship state from the Quartermaster
agent), the OOW must output a **natural-language helm order** ("30 degrees port", "1 degree port", "50
degrees starboard") and **engine order** ("full ahead", "full back", "continue course") -- not raw MOOS
calls (that translation is the later Actuator Agent's job).

**Deliberately not built in this pass**, analogous to how VHF's Track 2 (`pipeline/track2/build_vhf_*`)
was added only after Track 1 was working end-to-end. Concrete plan for the next pass:

1. **Scenario source**: reuse the 12 already-generated MOOS scenario geometries
   (`Data/OOW/OOW_Eval/generate_moos_scenarios.py`, `scenarios_manifest.json`) as *seed geometries*, and/or
   generate many more procedurally (bearing × range × own-speed × target-speed × encounter category),
   analogous to `build_vhf_colreg_scenarios.py`'s category list but for COLREG encounter types
   (head-on, crossing give-way/stand-on, overtaking, restricted visibility, narrow channel, vessel not
   under command, ...).
2. **New builder script** `pipeline/track2/build_oow_scenarios.py`: for each scenario, compute the
   COLREG-correct give-way/stand-on action from the closed-form geometry (reusing
   `OOW_Eval/score_scenario.py`'s CPA logic where possible), then render it as a fluent
   `{situation report -> helm order + engine order + rule justification}` pair -- same "fluent prose,
   never telegraphic label:value dumps" requirement the VHF `copilot-instructions.md` convention
   established for all training-data builders.
3. **Contamination filter**: embed against **both** `colreg_qa_500.json` (Track 1) and whatever new
   Track 2 held-out scenario file is created, exactly like VHF filters against both
   `vhf_gold_answers.json` and `vhf_colreg_scenarios.json`.
4. **Eval harness**: `pipeline/eval/eval_oow_scenarios.py` (analogous to `eval_colreg_scenarios.py`),
   scoring rule-citation correctness, give-way/stand-on correctness, and (new metric) whether the helm
   order's direction/magnitude and engine order are plausible for the stated geometry.
5. **MOOS-in-the-loop evaluation** (`colreg_llm_bridge.py` / `colreg_llm_bridge_paused.py`) stays a
   later phase, once the LLM reliably produces correct helm/engine orders in this text-only setting --
   confirmed as the intended order of operations in the earlier planning discussion for this domain.

Nothing in §14 onward below depends on Track 2 existing -- `train_sft.py`/`train_dpo.py`/
`train_reflection.py`'s `SFT_DATASETS`/`DPO_FILES`/`REFL_FILES` lists currently only reference Track 1
files for VHF's own Track 2; wiring OOW's future Track 2 files into those lists is a one-line addition
per file when the time comes.

---
## § 13.6 — Data-coverage gap analysis (reuse, no changes needed)

`pipeline/eval/analyze_gaps.py` is domain-agnostic by design (no VHF/COLREG field names hardcoded) --
point it at the baseline eval file from §5-7, grouped by `section_id` (here, COLREG rule reference).

In [ ]:
result = subprocess.run(
    [sys.executable, "-X", "utf8", "-m", "pipeline.eval.analyze_gaps",
     "--eval-file", str(paths.cache_dir / "eval_oow_qwen_base.jsonl"),
     "--group-by", "section_id"],
    cwd=W, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

---
## § 13.7 — Cross-source consistency check (framework reuse; fact-list is a TODO)

`pipeline/eval/check_consistency.py`'s *mechanism* (quote-scoped regex checks over every parsed JSON
section) is reusable, but its actual fact lists (`PHONETIC_ALPHABET`, `REPEAT_PROWORDS`,
`NUMBER_WORDS`) are VHF-specific and don't apply to COLREG text -- so this will currently report **zero
findings** for OOW, which is expected, not a sign everything is consistent. A COLREG-appropriate
Level-1 fact list (e.g. rule-number ↔ rule-title cross-references, light/shape/sound-signal
specifications that have exactly one correct answer) is a follow-up, not required before §14.

In [ ]:
result = subprocess.run(
    [sys.executable, "-X", "utf8", "-m", "pipeline.eval.check_consistency"],
    cwd=W, capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr)

---
## § 14 · Fine-tuning OOW-QWEN (3 stages)

Identical stages/hyperparameters to VHF -- `train_sft.py` → `train_dpo.py` → `train_reflection.py`,
each now domain-aware via `AgentPaths.from_env()` and saving to
`_models/OOW/oow_qwen_{sft,dpo,reflect}_lora/`. `SFT_DATASETS`/`DPO_FILES`/`REFL_FILES` in those three
scripts are keyed by `paths.domain`, so OOW's file lists (rule-text Track 1 from §12 + real-incident
excerpts from §12.7) are picked up automatically -- no CLI flags needed. Cross-track multi-hop
(rule text <-> incident) is already folded into `oow_multihop.jsonl` from §12. Track 2
(applied helm/engine-order decisions, §12.5/§12.6 above) is still not built, so this trains COLREG
rules/knowledge + real-incident reasoning only, until Track 2 data exists and is added to each
script's file list.

Stage 1 — SFT. Adapter ~50 MB.

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.train.train_sft"], cwd=W)

Stage 2 — DPO. Adapter ~25 MB.

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.train.train_dpo"], cwd=W)

Stage 3 — Reflection. Adapter ~25 MB.

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.train.train_reflection"], cwd=W)

---
## § 15 · Merge → OOW-QWEN (single deployable model)

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.train.merge_adapter"], cwd=W)

---
## § 16 · Evaluate OOW-QWEN on the 500 held-out COLREG Q&A (Track 1)

Closed-book (no prompt tricks) — measures what the fine-tune internalized. Scored with the claim-level
RAGAS suite v2 (AnswerCorrectness, CorpusGrounded, AnswerRelevancy, NumericF1, LitHit, Cover, fixed-weight
Composite — see VHF notebook §16 for definitions and why the old suite was replaced). Requires §12.9's
`colreg_qa_500_normalised_claims.json`; `--legacy` reproduces the old metrics. Summaries are stamped with
`metric_suite` — never compare `ragas_v2.0` numbers against `legacy_v1` numbers.


In [ ]:
subprocess.check_call([
    sys.executable, "-X", "utf8", "-m", "pipeline.eval.eval_finetuned",
    "--model", str(paths.domain_models_dir / "OOW-QWEN"),
    "--tag", "oow_qwen",
    "--gold-file", str(NORM_FILE),
], cwd=W)

---
## § 17 · Compression 1 — AWQ int4 quantization

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.compress.compress_quantize_awq"], cwd=W)

---
## § 18 · Compression 2 — Layer pruning (ShortGPT-style)

In [ ]:
subprocess.check_call([
    sys.executable, "-X", "utf8", "-m", "pipeline.compress.compress_prune", "--n-prune", "4",
], cwd=W)

---
## § 19 · Compression 3 — Knowledge Distillation → DistillOOW-QWEN

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.compress.compress_distill"], cwd=W)

---
## § 20 · Final comparison — all model tags, Track 1

Backfill eval for the AWQ/pruned/distilled tags, then compare everything side by side (same shape as
VHF's § 20 comparison table).

In [ ]:
tags_and_models = [
    ("oow_qwen_awq",      paths.domain_models_dir / "OOW-QWEN-awq-int4"),
    ("oow_qwen_pruned",   paths.domain_models_dir / "OOW-QWEN-pruned"),
    ("distill_oow_qwen",  paths.domain_models_dir / "DistillOOW-QWEN"),
]
for tag, model_dir in tags_and_models:
    if not model_dir.exists():
        print(f"  (skipping {tag} -- {model_dir} not found)")
        continue
    subprocess.check_call([
        sys.executable, "-X", "utf8", "-m", "pipeline.eval.eval_finetuned",
        "--model", str(model_dir), "--tag", tag, "--gold-file", str(NORM_FILE),
    ], cwd=W)

In [ ]:
import json
import pandas as pd

rows = []
for tag in ["oow_qwen_base", "oow_qwen", "oow_qwen_awq", "oow_qwen_pruned", "distill_oow_qwen"]:
    p = paths.cache_dir / f"eval_{tag}_summary.json"
    if not p.exists():
        continue
    s = json.loads(p.read_text())
    row = {"tag": tag, "n": s["n"], **s["means"]}
    if s.get("latency", {}).get("mean_s") is not None:
        row["latency_mean_s"] = s["latency"]["mean_s"]
    rows.append(row)

pd.set_option("display.width", 140)
print(pd.DataFrame(rows).to_string(index=False))